# 12 · The airframe: components, mass, CG, and mass closure as one equality

The `vehicle/` layer (630 lines) holds the physical airframe: lifting surfaces, fuselage, landing gear, systems, payload, fuel, nacelles,
fixed equipment, and the installed powertrain. `vehicle/aircraft.py` composes them and aggregates mass and CG. `weights/afdd.py` holds
the rotorcraft weight equations. This notebook builds an aircraft from parts, shows how every mass correlation is coupled back to the
take-off mass through one `StructuralDesignCondition`, and then closes mass and places the CG in one small Opti problem
(`examples/aircraft_mass_closure.py`), the second rung of the ladder after tutorial 10's powertrain point.

**In this notebook**
1. Lifting surfaces and the fuselage: parameterization, geometry through AeroSandbox, and a picture.
2. The structural design condition: how take-off mass feeds every correlation.
3. Raymer and AFDD mass correlations, and calibration factors.
4. The other items: gear, systems, payload, fuel, nacelles, equipment.
5. `PowertrainInstallation`: count × component mass at installed positions.
6. `Aircraft`: the mass breakdown and CG.
7. Mass closure and CG placement in one Opti; the growth factor.
8. What the Halo adds.

**Run time:** a few seconds.

In [1]:
from tutorial_helpers import *
import inspect
from dataclasses import replace, fields
from aircraft_closure.vehicle.surfaces import Wing, HorizontalTail, VerticalTail
from aircraft_closure.vehicle.fuselage import Fuselage
from aircraft_closure.vehicle.condition import StructuralDesignCondition
from aircraft_closure.vehicle.items import LandingGear, Systems, Payload, FuelLoad, FixedEquipment, Nacelles
from aircraft_closure.vehicle.powertrain_installation import PowertrainInstallation, InstalledInstance
from aircraft_closure.vehicle.aircraft import Aircraft, MassBreakdown

## 1. Surfaces and fuselage

Surfaces are parameterized by **planform area, aspect ratio and taper ratio** (the natural design variables), plus a root leading-edge
position. Span and chords follow: $b = \sqrt{SA}$, $c_r = 2S / (b(1 + \lambda))$, $c_t = \lambda c_r$. `to_asb()` hands the geometry to an
`asb.Wing`, which owns span, MAC, wetted area and so on (rule 1: don't duplicate AeroSandbox). All surfaces are unswept. The
`VerticalTail` is one fin with aspect ratio height²/area.

The fuselage is a constant section with conical nose and tail. The section is round, or a super-ellipse (`height_m`, `shape`; the Halo's
unpressurized box section is 1.68 m wide, 2.0 m deep, exponent 3.2).

In [2]:
wing = Wing(area_m2=12.0, aspect_ratio=9.0, taper_ratio=1.0, x_le_root_m=2.5)
tail_h, tail_v = HorizontalTail(), VerticalTail()
fuselage = Fuselage(length_m=7.0, diameter_m=1.2)
print(f"wing: span {wing.span_m():.2f} m, root chord {wing.chord_root_m():.3f} m, MAC {wing.to_asb().mean_aerodynamic_chord():.3f} m, "
      f"wetted {wing.to_asb().area('wetted'):.1f} m2")
print(f"vertical tail height {tail_v.height_m():.2f} m; fuselage wetted {fuselage.to_asb().area_wetted():.1f} m2, "
      f"volume {fuselage.to_asb().volume():.2f} m3")
check("span = sqrt(area x aspect ratio)", abs(wing.span_m() - (12.0 * 9.0)**0.5) < 1e-12)
opti = asb.Opti()
area = opti.variable(init_guess=12.0)
print("geometry with a variable area:", type(replace(wing, area_m2=area).to_asb().area()).__name__)

wing: span 10.39 m, root chord 1.155 m, MAC 1.155 m, wetted 25.1 m2
vertical tail height 1.60 m; fuselage wetted 19.6 m2, volume 5.13 m3
PASS  span = sqrt(area x aspect ratio)
geometry with a variable area: MX


## 2. The structural design condition

Empirical mass correlations depend on the design gross weight, the ultimate load factor and a cruise condition. Every one of them reads
these from one object:

In [3]:
print(inspect.getsource(StructuralDesignCondition))

@dataclass(frozen=True)
class StructuralDesignCondition:
    """Design gross mass and the flight condition the correlations are sized at.

    `mass_design_kg` is normally the caller's take-off mass variable, which is
    how the mass closure couples back into every correlation. Lift-to-drag is
    an assumption until aerodynamics exists (Tier 5); Raymer's fuselage
    correlation depends on it only weakly (exponent -0.072).
    """
    mass_design_kg: Any
    load_factor_ultimate: Any = 5.7
    velocity_cruise_m_s: Any = 60.0
    altitude_cruise_m: Any = 1000.0
    lift_to_drag_cruise: Any = 12.0

    def operating_point(self):
        return asb.OperatingPoint(atmosphere=asb.Atmosphere(altitude=self.altitude_cruise_m),
                                  velocity=self.velocity_cruise_m_s)



In a sizing problem `mass_design_kg` is **the take-off mass variable**. That single field is how the closure couples back into every
correlation: a heavier aircraft needs a heavier wing, which makes it heavier. In MDF terms this is the classic weight-loop coupling;
here it is just an argument (tutorial 03).

## 3. Mass correlations

`Wing`, the tails, the fuselage, the gear and the flight controls call AeroSandbox's **Raymer general-aviation** correlations
(*Aircraft Design: A Conceptual Approach*, sec. 15.3.3) on the AeroSandbox geometry. Each class has a `mass_factor`, a dimensionless
calibration multiplier. The wing also has a `mass_model` slot, filled in the Halo by the AFDD tiltrotor wing (`TiltrotorWingMassModel`,
sized by stiffness for whirl flutter, with a deep dive in `deep_dives/`).

In [4]:
condition = StructuralDesignCondition(mass_design_kg=1500.0)
for component in (wing, tail_h, tail_v):
    print(f"{type(component).__name__:<16} {component.get_mass_properties(condition).mass:7.1f} kg")
print(f"{'Fuselage':<16} {fuselage.get_mass_properties(condition, distance_wing_to_tail_m=3.8).mass:7.1f} kg")
masses = [wing.get_mass_properties(StructuralDesignCondition(mass_design_kg=m)).mass for m in (1000.0, 2000.0, 4000.0)]
print("wing mass at 1,000 / 2,000 / 4,000 kg design mass:", [f"{m:.0f}" for m in masses], "kg")

Wing               129.9 kg
HorizontalTail      11.7 kg
VerticalTail         8.8 kg
Fuselage            81.6 kg
wing mass at 1,000 / 2,000 / 4,000 kg design mass: ['106', '150', '210'] kg


The rotorcraft-specific groups use the **AFDD** equations (from NASA's NDARC): rotor blades and hub (`mass_rotor_group_afdd82_kg`), drive
system (tutorial 06), engine support, air induction and cowling (`Nacelles`), drive shafts (`InterconnectShaft`).

**Calibration.** The Halo multiplies each group by a factor that makes the model reproduce the Bell XV-15's published weight statement
at its design weight (`examples/xv15_reference.py: calibration_factors`). The factors carry everything the correlations miss (a tiltrotor
is not a GA airplane; the XV-15 is a crewed research aircraft):

In [5]:
from examples.xv15_reference import calibration_factors, compare_groups, published_groups
factors = calibration_factors()
predicted = compare_groups()
print(f"{'group':<18}{'predicted lb':>13}{'XV-15 lb':>10}{'factor':>8}")
for f in fields(factors):
    if f.name == "wing_tiltrotor":
        continue
    p, a = getattr(predicted, f.name), getattr(published_groups, f.name)
    print(f"{f.name:<18}{lb(p):13.0f}{lb(a):10.0f}{a / p:8.2f}")
print(f"{'wing (AFDD tiltrotor model)':<31}{'':>10}{factors.wing_tiltrotor:8.2f}")

group              predicted lb  XV-15 lb  factor
rotor                      1545      1070    0.69
wing                        452       873    1.93
tail                        186       209    1.13
fuselage                    696      1442    2.07
alighting_gear              585       508    0.87
flight_controls             247       934    3.79
powerplant                 1532      1754    1.14
transmission               1265      1263    1.00
wing (AFDD tiltrotor model)                  1.33


## 4. The other items

| Item | Mass | Notes |
|---|---|---|
| `LandingGear` | Raymer GA main + nose | fixed unless `is_retractable` |
| `Systems` | Raymer flight controls + installed avionics | `mass_factor` calibrates the manned-aircraft correlation |
| `Payload`, `FuelLoad`, `FixedEquipment` | given mass at a position | the fuel load is a design variable in sizing |
| `Nacelles` | AFDD engine support + air induction + cowling | optional geometry (bodies of revolution) for aerodynamics |
| `InterconnectShaft` | AFDD drive shaft | the XV-15's cross-shaft (not in the series hybrid) |

Every item returns an `asb.MassProperties` (mass and CG position), so they add with `+` (tutorial 01).

## 5. `PowertrainInstallation`

The installed powertrain places each **topology instance** at an (x, z) station and books `installation_factor × count × get_mass()`
there (mounts, cabling and cooling the component models omit). It refuses a location list that misses or invents an instance. An optional
`InstalledCooling` adds the heat exchanger. Symmetric copies sit at one x station, which is exact for CG when they are mirrored.

In [6]:
from examples.series_hybrid_point import build_reference_topology
topology = build_reference_topology(count_rotors=4)
try:
    PowertrainInstallation(topology, (InstalledInstance("motor", 3.0),))
except ValueError as error:
    print("incomplete locations ->", str(error)[:110])
x_rotor_m = wing.x_le_root_m + 0.25 * wing.chord_root_m()
locations = (InstalledInstance("turboshaft", 4.6, 0.1), InstalledInstance("generator", 4.1, 0.1),
             InstalledInstance("battery", 2.6, -0.3), InstalledInstance("motor", x_rotor_m, wing.z_m),
             InstalledInstance("gearbox", x_rotor_m, wing.z_m), InstalledInstance("propulsor", x_rotor_m, wing.z_m + 0.4))
installation = PowertrainInstallation(topology, locations, installation_factor=1.1)
for item in installation.get_instance_mass_properties():
    instance = topology.instances[item.instance_name]
    print(f"{item.instance_name:<12} x{instance.count}  {item.mass_properties.mass:6.1f} kg at x = {item.mass_properties.x_cg:.2f} m")

incomplete locations -> Locations must cover every instance exactly; missing ['battery', 'gearbox', 'generator', 'propulsor', 'turbosh
turboshaft   x1   330.0 kg at x = 4.60 m
generator    x1   110.0 kg at x = 4.10 m
battery      x1   176.0 kg at x = 2.60 m
motor        x4    88.0 kg at x = 2.79 m
gearbox      x4    44.0 kg at x = 2.79 m
propulsor    x4   132.0 kg at x = 2.79 m


## 6. `Aircraft`

`Aircraft` composes the twelve groups; `get_mass_breakdown(condition)` returns a `MassBreakdown` of `MassProperties`; `.total()` adds them;
`mass_empty_kg()` excludes payload and fuel. The assembly **never solves** (its docstring says so): closure is the caller's equality.
`examples/aircraft_mass_closure.py: build_reference_aircraft` assembles a tiltrotor-like layout around a topology. All rotors sit at the wing
quarter chord, so hover pitch trim with equal thrust needs the CG there:

In [7]:
from examples.aircraft_mass_closure import build_reference_aircraft
aircraft = build_reference_aircraft(x_le_wing_m=2.5, mass_payload_kg=300.0, count_rotors=4)
breakdown = aircraft.get_mass_breakdown(StructuralDesignCondition(mass_design_kg=1500.0))
for f in fields(MassBreakdown):
    item = getattr(breakdown, f.name)
    print(f"{f.name:<16}{item.mass:8.1f} kg   x_cg {item.x_cg if item.mass else float('nan'):6.2f} m")
total = breakdown.total()
print(f"{'sum':<16}{total.mass:8.1f} kg   x_cg {total.x_cg:6.2f} m   (design mass assumed: 1,500 kg)")
print(f"empty mass {breakdown.mass_empty_kg():.1f} kg")

wing               129.9 kg   x_cg   2.96 m
horizontal_tail     11.7 kg   x_cg   6.50 m
vertical_tail        8.8 kg   x_cg   6.41 m
fuselage            81.8 kg   x_cg   3.15 m
landing_gear        67.8 kg   x_cg   3.24 m
systems             66.2 kg   x_cg   1.50 m
powertrain         880.0 kg   x_cg   3.59 m
payload            300.0 kg   x_cg   3.00 m
fuel                 0.0 kg   x_cg    nan m
nacelles             0.0 kg   x_cg    nan m
drive_shaft          0.0 kg   x_cg    nan m
equipment            0.0 kg   x_cg    nan m
sum               1546.2 kg   x_cg   3.34 m   (design mass assumed: 1,500 kg)
empty mass 1246.2 kg


The parts add up to more than the 1,500 kg the correlations were evaluated at: this aircraft is not closed. And the CG is not under the
rotors.

## 7. Closure and CG placement in one Opti

Two unknowns, two equations: the take-off mass (which every correlation reads) and the wing position (which moves the rotors with it).
This is `solve_mass_closure`, written out:

In [8]:
opti = asb.Opti()
mass_takeoff_kg = opti.variable(init_guess=1800, scale=1000, lower_bound=100)
x_le_wing_m = opti.variable(init_guess=2.5, lower_bound=0.5, upper_bound=5.5)
aircraft = build_reference_aircraft(x_le_wing_m, mass_payload_kg=300.0, count_rotors=4)
breakdown = aircraft.get_mass_breakdown(StructuralDesignCondition(mass_design_kg=mass_takeoff_kg))
total = breakdown.total()
opti.subject_to([
    mass_takeoff_kg / total.mass == 1,                                        # mass closure
    total.x_cg == x_le_wing_m + 0.25 * aircraft.wing.chord_root_m(),           # hover pitch trim: CG under the rotors
])
solution = opti.solve(verbose=False)
print(f"closed: take-off {solution.value(mass_takeoff_kg):.2f} kg, wing leading edge at {solution.value(x_le_wing_m):.3f} m, "
      f"CG {solution.value(total.x_cg):.3f} m, {solution.stats()['iter_count']} iterations")
from examples.aircraft_mass_closure import solve_mass_closure
check("reproduces examples/aircraft_mass_closure.py", abs(solve_mass_closure().mass_takeoff_kg - solution.value(mass_takeoff_kg)) < 1e-6)

# Verify outside the optimizer: rebuild the aircraft with numbers and evaluate.
numeric = build_reference_aircraft(solution.value(x_le_wing_m), 300.0, 4)
numeric_total = numeric.get_mass_breakdown(StructuralDesignCondition(mass_design_kg=solution.value(mass_takeoff_kg))).total()
check("the numeric rebuild is closed and trimmed",
      abs(numeric_total.mass - solution.value(mass_takeoff_kg)) < 1e-6 and abs(numeric_total.x_cg - (solution.value(x_le_wing_m) + 0.25 * numeric.wing.chord_root_m())) < 1e-6)

closed: take-off 1552.39 kg, wing leading edge at 3.233 m, CG 3.521 m, 3 iterations
PASS  reproduces examples/aircraft_mass_closure.py
PASS  the numeric rebuild is closed and trimmed


That last check is a pattern used throughout the tests: **rebuild with numbers and re-evaluate**. Because every class works with numbers or
symbols, the solved design can be rebuilt as a plain numeric object and checked independently.

The growth factor of this closure (payload +1 kg):

In [9]:
growth = solve_mass_closure(mass_payload_kg=301.0).mass_takeoff_kg - solve_mass_closure(mass_payload_kg=300.0).mass_takeoff_kg
print(f"dMTOM/dpayload = {growth:.3f} kg/kg  (only structure grows here; the powertrain is fixed in this example)")
check("growth factor above 1", growth > 1)

dMTOM/dpayload = 1.110 kg/kg  (only structure grows here; the powertrain is fixed in this example)
PASS  growth factor above 1


## 8. What the Halo adds

`build_halo_aircraft` builds the same `Aircraft`, with:

- surfaces from design variables (wing area, aspect ratio optionally free, tail areas), a V-tail option, the AFDD tiltrotor wing;
- fuselage from the drawn layout (11 m, box section) with a layout-anchored factor (1.70 × raw Raymer, plan 037);
- XV-15 calibration factors on every group;
- nacelles at the tips holding rotor, motors and gearbox; turbogenerators in the fuselage behind the wing box;
- itemised fixed equipment for an uncrewed aircraft (`halo_equipment_items`);
- the heat exchanger as `InstalledCooling`.

Its closure line is the same: `mass_takeoff_kg / total.mass == 1`.

## Summary

- Surfaces are area / aspect ratio / taper ratio; geometry goes through AeroSandbox; everything is differentiable.
- One `StructuralDesignCondition` carries the design mass (the take-off mass variable) into every correlation.
- Raymer GA and AFDD correlations, each multiplied by an XV-15 calibration factor in the Halo.
- `PowertrainInstallation` books count × component mass at installed positions; `Aircraft` adds `MassProperties`; it never solves.
- Closure is one normalized equality; trim placement is another; verify by rebuilding with numbers.

## Check yourself

<details><summary>1. Where would you put a "structural weight growth allowance" of 5 % on the airframe?</summary>

As a factor on the airframe groups (the `mass_factor` fields) or as a `FixedEquipment` item of 5 % of the structure expression. Either way it enters the closure equality and the optimizer resizes around it. The Halo's OEW tutorial (deep dives) separates growth allowance from uncertainty.
</details>

<details><summary>2. Why is the fuselage's L/D in the structural condition, and what supplies it in a sizing problem?</summary>

Raymer's fuselage correlation depends weakly on cruise L/D (exponent -0.072). In `aircraft_mass_closure.py` it is assumed (12); in `cruise_closure.py` and the coupled sizing it is the solved cruise point's L/D, an expression, so aerodynamics and weights couple through it.
</details>

In [10]:
check_summary()

4 of 4 checks passed
